<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/22_rag_caching/rag_caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers scikit-learn --quiet

In [ ]:
import time
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load models
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    "Paris is the capital of France.",
    "France is located in Europe.",
    "Berlin is the capital of Germany.",
    "Python is a programming language."
]

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

embeddings = embed_model.encode(documents)

In [ ]:
def rag_pipeline(query):

    query_embedding = embed_model.encode([query])
    query_tfidf = vectorizer.transform([query])

    semantic_scores = cosine_similarity(query_embedding, embeddings)[0]
    keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

    hybrid_scores = 0.5 * semantic_scores + 0.5 * keyword_scores

    best_index = hybrid_scores.argmax()
    context = documents[best_index]

    prompt = f"""
Answer the question using the context.

Context: {context}

Question: {query}
"""

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [ ]:
cache = {}

def rag_with_cache(query):

    # Check cache
    if query in cache:
        print("⚡ Cache Hit")
        return cache[query]

    print("⏳ Cache Miss → Running RAG")

    # Run pipeline
    answer = rag_pipeline(query)

    # Store result
    cache[query] = answer

    return answer

In [9]:
query = "What is the capital of France?"

print("First Run:")
print(rag_with_cache(query))

print("\nSecond Run:")
print(rag_with_cache(query))

First Run:
⚡ Cache Hit
Paris

Second Run:
⚡ Cache Hit
Paris


In [10]:
query = "What is the capital of Germany?"

# Without cache
start = time.time()
rag_pipeline(query)
end = time.time()

print("Without Cache Time:", end - start)

# With cache (first time)
start = time.time()
rag_with_cache(query)
end = time.time()

print("With Cache (first call):", end - start)

# With cache (second time)
start = time.time()
rag_with_cache(query)
end = time.time()

print("With Cache (cached):", end - start)

Without Cache Time: 0.7951459884643555
⚡ Cache Hit
With Cache (first call): 8.249282836914062e-05
⚡ Cache Hit
With Cache (cached): 6.723403930664062e-05
